In [1]:
# ============================================================
# TEMPORAL PPE VIOLATION INTELLIGENCE SYSTEM
# ============================================================

# ============================================================
# IMPORTS
# ============================================================

from ultralytics import YOLO

import cv2
import numpy as np
import time
import os

from pathlib import Path
from collections import defaultdict

# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path("../")

MODEL_PATH = PROJECT_ROOT / "models" / "trained" / "ppe_yolo11m_best.pt"

INPUT_VIDEO = PROJECT_ROOT / "videos" / "input" / "input.mp4"

OUTPUT_VIDEO = PROJECT_ROOT / "videos" / "output" / "05_temporal_violations_demo.mp4"

ALERT_DIR = PROJECT_ROOT / "alerts"

# ============================================================
# CREATE ALERT DIRECTORY
# ============================================================

ALERT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# LOAD MODEL
# ============================================================

model = YOLO(str(MODEL_PATH))

print("=" * 60)
print("MODEL LOADED SUCCESSFULLY")
print("=" * 60)

# ============================================================
# OPEN VIDEO
# ============================================================

cap = cv2.VideoCapture(str(INPUT_VIDEO))

if not cap.isOpened():
    raise ValueError(f"Cannot open video: {INPUT_VIDEO}")

# ============================================================
# VIDEO PROPERTIES
# ============================================================

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fps = int(cap.get(cv2.CAP_PROP_FPS))

print(f"Resolution : {frame_width} x {frame_height}")
print(f"FPS        : {fps}")

# ============================================================
# OUTPUT VIDEO WRITER
# ============================================================

OUTPUT_VIDEO.parent.mkdir(
    parents=True,
    exist_ok=True
)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    str(OUTPUT_VIDEO),
    fourcc,
    fps,
    (frame_width, frame_height)
)

# ============================================================
# CLASS NAMES
# ============================================================

CLASS_NAMES = model.names

# ============================================================
# COLORS
# ============================================================

SAFE_COLOR = (0, 255, 0)

WARNING_COLOR = (0, 165, 255)

DANGER_COLOR = (0, 0, 255)

CRITICAL_COLOR = (0, 0, 180)

# ============================================================
# TRACK HISTORY
# ============================================================

track_history = {}

# ============================================================
# TEMPORAL MEMORY
# ============================================================

violation_memory = defaultdict(dict)

# ============================================================
# ALERT SETTINGS
# ============================================================

HELMET_ALERT_TIME = 5

VEST_ALERT_TIME = 5

CRITICAL_ALERT_TIME = 3

# ============================================================
# PERFORMANCE VARIABLES
# ============================================================

prev_time = time.time()

frame_count = 0

# ============================================================
# IOU FUNCTION
# ============================================================

def calculate_iou(boxA, boxB):

    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])

    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)

    if interArea == 0:
        return 0

    boxAArea = (
        (boxA[2] - boxA[0]) *
        (boxA[3] - boxA[1])
    )

    boxBArea = (
        (boxB[2] - boxB[0]) *
        (boxB[3] - boxB[1])
    )

    iou = interArea / float(
        boxAArea + boxBArea - interArea
    )

    return iou

# ============================================================
# ALERT FUNCTION
# ============================================================

def save_alert(frame, worker_id, alert_type):

    timestamp = int(time.time())

    filename = (
        f"worker_{worker_id}_"
        f"{alert_type}_"
        f"{timestamp}.jpg"
    )

    save_path = ALERT_DIR / filename

    cv2.imwrite(
        str(save_path),
        frame
    )

    print("=" * 60)
    print("ALERT GENERATED")
    print("=" * 60)

    print(f"Worker ID : {worker_id}")
    print(f"Alert     : {alert_type}")
    print(f"Saved To  : {save_path}")

# ============================================================
# INFERENCE LOOP
# ============================================================

print("\nStarting temporal violation intelligence...\n")

while True:

    success, frame = cap.read()

    if not success:
        print("\nVideo processing completed.")
        break

    # --------------------------------------------------------
    # CURRENT TIME
    # --------------------------------------------------------

    current_time = time.time()

    # --------------------------------------------------------
    # YOLO + BYTE TRACKING
    # --------------------------------------------------------

    results = model.track(

        source=frame,

        persist=True,

        tracker="bytetrack.yaml",

        conf=0.40,

        imgsz=960,

        verbose=False,

        device=0
    )

    result = results[0]

    annotated_frame = frame.copy()

    # ========================================================
    # STORE DETECTIONS
    # ========================================================

    persons = []

    helmets = []

    vests = []

    # ========================================================
    # EXTRACT DETECTIONS
    # ========================================================

    if result.boxes is not None and result.boxes.id is not None:

        boxes = result.boxes.xyxy.cpu().numpy()

        class_ids = result.boxes.cls.cpu().numpy().astype(int)

        confidences = result.boxes.conf.cpu().numpy()

        track_ids = result.boxes.id.cpu().numpy().astype(int)

        for box, class_id, conf, track_id in zip(
            boxes,
            class_ids,
            confidences,
            track_ids
        ):

            class_name = CLASS_NAMES[class_id]

            x1, y1, x2, y2 = map(int, box)

            detection = {
                "box": [x1, y1, x2, y2],
                "conf": conf,
                "track_id": track_id
            }

            if class_name == "Person":
                persons.append(detection)

            elif class_name == "Hardhat":
                helmets.append(detection)

            elif class_name == "Safety Vest":
                vests.append(detection)

    # ========================================================
    # PPE ANALYSIS
    # ========================================================

    for person in persons:

        px1, py1, px2, py2 = person["box"]

        worker_id = person["track_id"]

        # ----------------------------------------------------
        # BODY REGIONS
        # ----------------------------------------------------

        person_height = py2 - py1

        head_region = [

            px1,
            py1,
            px2,
            py1 + int(person_height * 0.30)
        ]

        torso_region = [

            px1,
            py1 + int(person_height * 0.30),
            px2,
            py1 + int(person_height * 0.70)
        ]

        # ----------------------------------------------------
        # PPE FLAGS
        # ----------------------------------------------------

        has_helmet = False

        has_vest = False

        # ----------------------------------------------------
        # HELMET CHECK
        # ----------------------------------------------------

        for helmet in helmets:

            iou = calculate_iou(
                head_region,
                helmet["box"]
            )

            if iou > 0.01:
                has_helmet = True
                break

        # ----------------------------------------------------
        # VEST CHECK
        # ----------------------------------------------------

        for vest in vests:

            iou = calculate_iou(
                torso_region,
                vest["box"]
            )

            if iou > 0.01:
                has_vest = True
                break

        # ====================================================
        # TEMPORAL VIOLATION LOGIC
        # ====================================================

        status = "SAFE"

        color = SAFE_COLOR

        # ----------------------------------------------------
        # NO HELMET
        # ----------------------------------------------------

        if not has_helmet:

            if "helmet_start" not in violation_memory[worker_id]:

                violation_memory[worker_id]["helmet_start"] = current_time

            duration = (
                current_time -
                violation_memory[worker_id]["helmet_start"]
            )

            status = f"NO HELMET ({duration:.1f}s)"

            color = DANGER_COLOR

            # ALERT
            if duration > HELMET_ALERT_TIME:

                status = f"HELMET ALERT ({duration:.1f}s)"

                color = CRITICAL_COLOR

                if not violation_memory[worker_id].get(
                    "helmet_alert_sent",
                    False
                ):

                    save_alert(
                        annotated_frame,
                        worker_id,
                        "NO_HELMET"
                    )

                    violation_memory[worker_id][
                        "helmet_alert_sent"
                    ] = True

        else:

            violation_memory[worker_id].pop(
                "helmet_start",
                None
            )

            violation_memory[worker_id].pop(
                "helmet_alert_sent",
                None
            )

        # ----------------------------------------------------
        # NO VEST
        # ----------------------------------------------------

        if not has_vest:

            if "vest_start" not in violation_memory[worker_id]:

                violation_memory[worker_id]["vest_start"] = current_time

            duration = (
                current_time -
                violation_memory[worker_id]["vest_start"]
            )

            status = f"NO VEST ({duration:.1f}s)"

            color = WARNING_COLOR

            # ALERT
            if duration > VEST_ALERT_TIME:

                status = f"VEST ALERT ({duration:.1f}s)"

                color = DANGER_COLOR

                if not violation_memory[worker_id].get(
                    "vest_alert_sent",
                    False
                ):

                    save_alert(
                        annotated_frame,
                        worker_id,
                        "NO_VEST"
                    )

                    violation_memory[worker_id][
                        "vest_alert_sent"
                    ] = True

        else:

            violation_memory[worker_id].pop(
                "vest_start",
                None
            )

            violation_memory[worker_id].pop(
                "vest_alert_sent",
                None
            )

        # ----------------------------------------------------
        # CRITICAL: NO PPE
        # ----------------------------------------------------

        if not has_helmet and not has_vest:

            if "critical_start" not in violation_memory[worker_id]:

                violation_memory[worker_id]["critical_start"] = current_time

            duration = (
                current_time -
                violation_memory[worker_id]["critical_start"]
            )

            status = f"CRITICAL ({duration:.1f}s)"

            color = CRITICAL_COLOR

            # ALERT
            if duration > CRITICAL_ALERT_TIME:

                if not violation_memory[worker_id].get(
                    "critical_alert_sent",
                    False
                ):

                    save_alert(
                        annotated_frame,
                        worker_id,
                        "CRITICAL_NO_PPE"
                    )

                    violation_memory[worker_id][
                        "critical_alert_sent"
                    ] = True

        else:

            violation_memory[worker_id].pop(
                "critical_start",
                None
            )

            violation_memory[worker_id].pop(
                "critical_alert_sent",
                None
            )

        # ====================================================
        # DRAW PERSON
        # ====================================================

        cv2.rectangle(
            annotated_frame,
            (px1, py1),
            (px2, py2),
            color,
            3
        )

        label = f"Worker {worker_id} | {status}"

        cv2.putText(
            annotated_frame,
            label,
            (px1, py1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            color,
            2
        )

        # ====================================================
        # TRAJECTORY
        # ====================================================

        center_x = int((px1 + px2) / 2)
        center_y = int((py1 + py2) / 2)

        if worker_id not in track_history:
            track_history[worker_id] = []

        track_history[worker_id].append(
            (center_x, center_y)
        )

        if len(track_history[worker_id]) > 30:
            track_history[worker_id].pop(0)

        points = track_history[worker_id]

        for i in range(1, len(points)):

            cv2.line(
                annotated_frame,
                points[i - 1],
                points[i],
                color,
                2
            )

    # ========================================================
    # FPS
    # ========================================================

    fps_value = 1 / (current_time - prev_time)

    prev_time = current_time

    cv2.putText(
        annotated_frame,
        f"FPS: {fps_value:.2f}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )

    # ========================================================
    # FRAME COUNT
    # ========================================================

    frame_count += 1

    cv2.putText(
        annotated_frame,
        f"Frame: {frame_count}",
        (20, 80),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255, 255, 0),
        2
    )

    # ========================================================
    # DISPLAY
    # ========================================================

    cv2.imshow(
        "Temporal PPE Violation Intelligence",
        annotated_frame
    )

    # ========================================================
    # SAVE FRAME
    # ========================================================

    out.write(annotated_frame)

    # ========================================================
    # EXIT
    # ========================================================

    key = cv2.waitKey(1)

    if key == ord("q"):
        print("\nStopped by user.")
        break

# ============================================================
# RELEASE RESOURCES
# ============================================================

cap.release()

out.release()

cv2.destroyAllWindows()

# ============================================================
# DONE
# ============================================================

print("=" * 60)
print("TEMPORAL VIOLATION ANALYSIS COMPLETED")
print("=" * 60)

print(f"\nSaved Output:\n{OUTPUT_VIDEO}")

print(f"\nAlerts Saved In:\n{ALERT_DIR}")

MODEL LOADED SUCCESSFULLY
Resolution : 1920 x 1080
FPS        : 30

Starting temporal violation intelligence...

ALERT GENERATED
Worker ID : 42
Alert     : CRITICAL_NO_PPE
Saved To  : C:\Users\VANSH\OneDrive - IITRAM\Desktop\Projects\PPE_Project\alerts\worker_42_CRITICAL_NO_PPE_1778864973.jpg
ALERT GENERATED
Worker ID : 50
Alert     : CRITICAL_NO_PPE
Saved To  : C:\Users\VANSH\OneDrive - IITRAM\Desktop\Projects\PPE_Project\alerts\worker_50_CRITICAL_NO_PPE_1778864976.jpg
ALERT GENERATED
Worker ID : 50
Alert     : NO_HELMET
Saved To  : C:\Users\VANSH\OneDrive - IITRAM\Desktop\Projects\PPE_Project\alerts\worker_50_NO_HELMET_1778864978.jpg
ALERT GENERATED
Worker ID : 50
Alert     : NO_VEST
Saved To  : C:\Users\VANSH\OneDrive - IITRAM\Desktop\Projects\PPE_Project\alerts\worker_50_NO_VEST_1778864978.jpg
ALERT GENERATED
Worker ID : 85
Alert     : NO_VEST
Saved To  : C:\Users\VANSH\OneDrive - IITRAM\Desktop\Projects\PPE_Project\alerts\worker_85_NO_VEST_1778864992.jpg

Video processing complete